<a href="https://colab.research.google.com/github/smerashanmugan/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** item from an order.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [10]:
import pandas as pd
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

orders
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


The first line that I wrote is taking the nested JSON data and turning it into a flat table, orders. Then the next line is telling python to use the our orders data. The next line goes into each line and makes each item itws own row. The meta line then keeps the order number and the vendor ID with each item. Finally at the end, orders displays the finished table.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [14]:
vendors = pd.json_normalize(vendors_json)

joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

joined

unmatched = joined[joined['_merge'] == 'left_only']

unmatched

unmatched[['vendor_id', 'qty']]

,vendor_id,qty
3,V-07,3


For this question, for the first line of code I turned the vendor JSON data into a flat table called vendors which I then combined with the orders table in the second line of code. The on='vendor id' line is matching the two tables using the vendor id column. The how= 'left' line is using a left join so that all of the orders are still in the table even if they had no match. The indicator=True line tells me whether each row was matched or not. At the bottom, the joined displays the combined table. The unmatched = joined line filters the table so that only the orders that did not have a mach are shown. Then unmatched tells us which row was unmatched. Finally, the bottom line tells us the vendor ID and the quantity of the unmatched.  

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [6]:
joined['name'] = joined['name'].fillna('Unknown vendor')

total_by_vendor = joined.groupby('name')['qty'].sum()

total_by_vendor
# The unmatched vendor is labeled as "Unknown vendor" so the quantity for that is not lost.

,qty
name,
Cav Merch,1
Hoos Burgers,3
Unknown vendor,3


The first line in the code is replacing the missing vendors's name with unknown vendor. The second line of code is grouping the rows by vendor name and adding up the quanitity for each vendor. Finally the last line is showing the total quantitiy for each vendor.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [7]:
zone_totals = joined.groupby('zone')['qty'].sum().reset_index()

print(zone_totals)
print('Total:', zone_totals['qty'].sum())

  zone  qty
0    A    4
Total: 4


The total is 4 units because the unmatched vendor has no zone so the 3 units from that vendor are not included. So the first line of code is grouping the data by zone and adding up the quantity sold in each zone and the result is turned into a table. The second line of code is showing the total quantity for each zone. The final line of code is adding the zone totals together so we can see how many units were included.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [8]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

has_lines = [x for x in ragged_json if 'lines' in x]

ragged = pd.json_normalize(
    has_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

print(ragged)
# TODO

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


For the first line of code that I had inputted, it keeps only the orders that actually have a lines section to prevent the missing lines from causing an error. The second line flattens the remaining JSON into a table and the third,fourth, and fifth line filter the data that contains lines, make each line into its own row, and eep the order number and vendor ID with each line item respectively. The print statement then shows us the finished table.

### Q6 — Validate

In [15]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is the useful shape because it makes it easier to look at each item separately and allows us to calculate things easier such as how many of each item there are.
b) A missing quantity means that the quantity is unknown while zero means we know that the quantitiy was 0. If missing values are treated as zero, then any analysis performed on the data would be inaccurate because the unknown is now treated as real data.